# Smoke-эксперимент: фильтр SMS-спама

## tl;dr

Из 170 синтетических строк после нормализованной дедупликации осталось 142 сообщения. Validation выбрал word TF-IDF и порог 0.7110 с precision/recall 1.0000/1.0000; на smoke-test PR-AUC, precision и recall также равны 1.0000. После простой обфускации detection rate спама падает с 1.0 до 0.2 — важное ограничение, несмотря на идеальные synthetic-метрики.

## Context & Methods

Notebook запускает production `train`, `evaluate` и scoring. Нормализованные дубликаты удаляются до split; на validation сравниваются word, char и combined TF-IDF. Порог выбирается по требованию минимальной precision 0.95 с максимальным доступным recall.

### Key Assumptions

- высокая precision важнее recall в учебном сценарии ошибочной блокировки;
- дубликаты определяются casefold и нормализацией пробелов, near-duplicates остаются;
- synthetic prevalence и шаблоны не отражают UCI или production;
- обфускация `free→fr33`, `win→w1n` — узкая диагностика, не полная adversarial-оценка.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sms_spam_filter.data import make_smoke_data, read_input
from sms_spam_filter.evaluate import evaluate
from sms_spam_filter.predict import score_messages
from sms_spam_filter.train import train

RUN_DIR = PROJECT_ROOT / "artifacts" / "notebook_smoke"
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = RUN_DIR / "sms.csv"
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/dviakhrameev/Documents/helen/sms-spam-filter


## Data

Генератор создаёт ham/spam-шаблоны и два явных дубликата; дополнительные совпадения возникают из-за повторяющихся шаблонов. Публичный loader выполняет schema validation и дедупликацию до моделирования.

In [2]:
raw_smoke = make_smoke_data(ham_count=120, spam_count=48)
raw_smoke.to_csv(DATA_PATH, index=False)
validated = read_input(DATA_PATH)
data_summary = {
    "raw_rows": len(raw_smoke),
    "deduplicated_rows": len(validated),
    "removed_duplicates": len(raw_smoke) - len(validated),
    "spam_rate_after_dedup": round(float(validated["is_spam"].mean()), 4),
}
print(json.dumps(data_summary, ensure_ascii=False, indent=2))
validated[["label", "text", "is_spam"]].head(4)

{
  "raw_rows": 170,
  "deduplicated_rows": 142,
  "removed_duplicates": 28,
  "spam_rate_after_dedup": 0.338
}


,label,text,is_spam
0,ham,"Meeting moved to 8:00, see you in room 1",0
1,ham,Can you call me after work about ticket 1?,0
2,ham,Your appointment is confirmed for day 3,0
3,ham,Please bring milk and bread when you come home 3,0


## Results

Train сохраняет TF-IDF pipeline, precision-порог и held-out predictions. Evaluate повторно проверяет API на полном smoke-файле; single-message scoring демонстрирует CLI-эквивалент без сети.

In [3]:
train_metrics = train(DATA_PATH, RUN_DIR, min_precision=0.95, seed=42)
api_evaluation = evaluate(DATA_PATH, RUN_DIR / "model.joblib")
example_scores = score_messages(
    pd.Series(["Lunch at noon", "FREE prize, call now!"], dtype="string"),
    RUN_DIR / "model.joblib",
)
result_summary = {
    "selected_model": train_metrics["selected_model"],
    "split": train_metrics["split"],
    "precision_constraint": train_metrics["precision_constraint"],
    "held_out_test": {
        "average_precision": round(train_metrics["test"]["average_precision"], 4),
        "precision": round(train_metrics["test"]["precision"], 4),
        "recall": round(train_metrics["test"]["recall"], 4),
    },
    "obfuscation_test": train_metrics["obfuscation_test"],
    "full_smoke_api_check_ap": round(api_evaluation["average_precision"], 4),
}
print(json.dumps(result_summary, ensure_ascii=False, indent=2))
example_scores

{
  "selected_model": "word_tfidf",
  "split": {
    "train": 84,
    "validation": 29,
    "test": 29
  },
  "precision_constraint": {
    "requested": 0.95,
    "threshold": 0.7110173690476959,
    "precision": 1.0,
    "recall": 1.0,
    "constraint_met": true
  },
  "held_out_test": {
    "average_precision": 1.0,
    "precision": 1.0,
    "recall": 1.0
  },
  "obfuscation_test": {
    "spam_messages": 10,
    "original_detection_rate": 1.0,
    "obfuscated_detection_rate": 0.2,
    "mean_probability_shift": -0.06692261245352651
  },
  "full_smoke_api_check_ap": 1.0
}


,text,spam_probability,is_spam,threshold
0,Lunch at noon,0.408095,0,0.711017
1,"FREE prize, call now!",0.734687,1,0.711017


In [4]:
assert train_metrics["precision_constraint"]["constraint_met"]
assert len(validated) < len(raw_smoke)
assert example_scores["spam_probability"].between(0, 1).all()
assert (RUN_DIR / "model.joblib").exists()
print("Deduplication, threshold and artifact checks: OK")

Deduplication, threshold and artifact checks: OK


## Takeaways

- Дедупликация убрала 28 из 170 smoke-строк до split, предотвращая прямое повторение текстов.
- Требование validation precision 0.95 выполнено, но идеальные test-метрики вызваны простотой synthetic-шаблонов.
- Падение detection rate после обфускации до 0.2 показывает, что даже word/char experiments нужно проверять на более разнообразных и временно новых данных.